# 06 Vocabulary threshold (CURE only)

In [1]:
import sys
import time
from pathlib import Path
import numpy as np
import pandas as pd
import scipy.sparse as sp
import matplotlib.pyplot as plt
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import normalize
sys.path.append("..")          # cf_common.py and cure.py are in the folder above

from cure import cure
from cf_common import Data, evaluate_clustering, save_results, assign_to_nearest

DATA_FILE = Path("../data/user_top_tracks.csv")
N_USERS = None                  # None = all users, e.g. 20_000 for a quick test
MIN_USERS_VALUES = [20, 10, 5]     # vocabulary thresholds to try (20 = the main results)

# the same settings as 00_prepare_data and 02_cure
N_HIDDEN = 10
EMBEDDING_DIM = 8
N_EVAL_USERS = 10_000
SEED = 579
K_VALUES = [10, 20, 40, 80]
SAMPLE_SIZE = 10_000
N_REPRESENTATIVES = 8
SHRINK = 0.8
MIN_CLUSTER_SIZE = 10

## Load and split (same as `00_prepare_data`)
The split doesn't depend on the vocabulary, so it is done once.

In [2]:
start = time.perf_counter()
df = pd.read_csv(DATA_FILE,
                 usecols=["user_id", "track_name", "artist_name", "playcount"],
                 nrows=N_USERS * 50 if N_USERS else None,
                 dtype={"user_id": "int32", "playcount": "int32", "track_name": "category", "artist_name": "category"},
                 keep_default_na=False, na_values=[""])
df = df.dropna(subset=["track_name", "artist_name"])
df = df[df["artist_name"] != "[unknown]"]
df["user"] = df["user_id"].astype("category").cat.codes.astype("int32")
df["track"] = df.groupby(["track_name", "artist_name"], observed=True).ngroup().astype("int32")   # a track is a (title, artist) pair
n_users = df["user"].max() + 1
n_all_tracks = df["track"].max() + 1

# hide 10 random tracks per user (only users with at least 20 tracks)
df = df.sample(frac=1, random_state=SEED)
position = df.groupby("user").cumcount()
n_tracks = df.groupby("user")["track"].transform("size")
df["hidden"] = (position < N_HIDDEN) & (n_tracks >= 2 * N_HIDDEN)
train = df[~df["hidden"]]

# the 10,000 test users and their hidden tracks
rng = np.random.default_rng(SEED)
users_with_hidden = np.unique(df.loc[df["hidden"], "user"])
eval_users = np.sort(rng.choice(users_with_hidden, size=min(N_EVAL_USERS, len(users_with_hidden)), replace=False))
hidden_tracks = df[df["hidden"] & df["user"].isin(eval_users)].sort_values("user")["track"].to_numpy()

if N_USERS is None:       # check: the same test users as 00_prepare_data
    assert np.array_equal(eval_users, np.load("cache/eval_users.npy"))

print(f"{n_users:,} users, {n_all_tracks:,} tracks, {len(train):,} training rows "
      f"({time.perf_counter() - start:.0f}s)")

476,451 users, 3,325,347 tracks, 19,050,829 training rows (69s)


## Matrix and embedding for one threshold
The same steps as `00_prepare_data`, only `min_users` changes.

In [3]:
def build_data(min_users):
    users_per_track = np.bincount(train["track"], minlength=n_all_tracks)
    vocabulary = np.flatnonzero(users_per_track >= min_users)     # track ids that become columns
    column_of_track = np.full(n_all_tracks, -1)                   # track id -> column (-1 = not used)
    column_of_track[vocabulary] = np.arange(len(vocabulary))

    rows = train[column_of_track[train["track"]] >= 0]
    ratings = sp.csr_matrix(
        (np.log1p(rows["playcount"].to_numpy()).astype(np.float32),
         (rows["user"].to_numpy(), column_of_track[rows["track"]])),
        shape=(n_users, len(vocabulary)))

    svd = TruncatedSVD(n_components=EMBEDDING_DIM, random_state=SEED)
    embedding = normalize(svd.fit_transform(normalize(ratings)))
    hidden_items = column_of_track[hidden_tracks].reshape(len(eval_users), N_HIDDEN)

    info = {"min users per track": min_users,
            "tracks in vocabulary": len(vocabulary),
            "training rows dropped": len(train) - len(rows),
            "training rows dropped %": 100 * (len(train) - len(rows)) / len(train),
            "non-zeros (millions)": ratings.nnz / 1e6,
            "users with empty vector": int((ratings.getnnz(axis=1) == 0).sum()),
            "hidden recommendable %": 100 * (hidden_items >= 0).mean(),
            "explained variance %": 100 * svd.explained_variance_ratio_.sum()}
    return Data(ratings, embedding, eval_users, hidden_items), info

## CURE for every threshold
Same 10,000-user sample as `02_cure`.

In [4]:
sample = np.random.default_rng(579).choice(n_users, size=min(SAMPLE_SIZE, n_users), replace=False)

summaries, per_user, vocabulary_info = [], [], []
for min_users in MIN_USERS_VALUES:
    data, info = build_data(min_users)
    vocabulary_info.append(info)
    print(f"--- at least {min_users} users: {info['tracks in vocabulary']:,} tracks, "
          f"{info['hidden recommendable %']:.1f}% of the hidden tracks can be recommended")

    start = time.perf_counter()
    representatives = cure(data.embedding[sample], K_VALUES, N_REPRESENTATIVES, SHRINK, MIN_CLUSTER_SIZE)
    cure_seconds = time.perf_counter() - start
    for k in K_VALUES:
        start = time.perf_counter()
        rep_points, rep_owner = representatives[k]
        labels = rep_owner[assign_to_nearest(data.embedding, rep_points)]
        fit_seconds = cure_seconds + time.perf_counter() - start

        summary, users = evaluate_clustering(data, "cure", k, labels, fit_seconds,
                                             extra={"min users per track": min_users})
        summaries.append(summary)
        per_user.append(users)

results = save_results("vocabulary_threshold", summaries, per_user, folder="results/analyses")
vocabulary_info = pd.DataFrame(vocabulary_info)
vocabulary_info.to_csv("results/analyses/vocabulary_threshold_info.csv", index=False)

--- at least 20 users: 120,650 tracks, 67.7% of the hidden tracks can be recommended


cure             k=10   P@10 = 0.1257   clustering   39.2s   CF   32.6s


cure             k=20   P@10 = 0.1228   clustering   39.2s   CF   23.5s


cure             k=40   P@10 = 0.1202   clustering   39.4s   CF   22.5s


cure             k=80   P@10 = 0.1153   clustering   39.8s   CF   25.8s


--- at least 10 users: 221,287 tracks, 74.4% of the hidden tracks can be recommended


... (6 outputs left out)

cure             k=20   P@10 = 0.1259   clustering   49.1s   CF   57.8s


cure             k=40   P@10 = 0.1241   clustering   49.2s   CF   56.8s


cure             k=80   P@10 = 0.1166   clustering   49.6s   CF   50.6s


saved results\analyses\vocabulary_threshold.csv and results\analyses\vocabulary_threshold_per_user.csv


## Results

In [5]:
vocabulary_info.round(2)

,min users per track,tracks in vocabulary,training rows dropped,training rows dropped %,non-zeros (millions),users with empty vector,hidden recommendable %,explained variance %
0,20,120650,6004246,31.52,13.05,2948,67.72,1.23
1,10,221287,4649529,24.41,14.40,1535,74.40,1.19
2,5,409289,3433359,18.02,15.62,742,80.20,1.16


In [6]:
results[["min users per track", "k", "clusters", "largest cluster %", "avg neighbour pool",
         "P@5", "P@10", "R@20", "clustering s", "CF s (all users, estimated)"]].round(4)

,min users per track,k,clusters,largest cluster %,avg neighbour pool,P@5,P@10,R@20,clustering s,"CF s (all users, estimated)"
0,20,10,9,34.9316,99910.9724,0.1794,0.1257,0.1650,39.1663,1555.1662
1,20,20,14,32.9929,81934.1852,0.1756,0.1228,0.1618,39.2252,1121.2219
2,20,40,24,31.0462,71129.9358,0.1719,0.1202,0.1598,39.4323,1070.3494
3,20,80,46,27.5588,51653.9578,0.1628,0.1153,0.1536,39.8284,1226.9010
4,10,10,10,32.4260,103961.1281,0.1838,0.1286,0.1696,64.3979,1759.4347
5,10,20,15,31.1579,82582.7177,0.1785,0.1254,0.1660,64.4251,1673.0767
6,10,40,24,25.5552,59995.6766,0.1718,0.1213,0.1615,64.6250,1572.6722
7,10,80,43,18.5494,32982.6272,0.1613,0.1148,0.1540,65.0418,1894.3302
8,5,10,9,41.8196,117123.9287,0.1844,0.1292,0.1716,48.9894,2962.7065
9,5,20,12,35.0873,88044.4348,0.1796,0.1259,0.1676,49.0610,2752.3174


In [7]:
# the user filter: training rows each threshold drops, and P@10 for every k
p10 = results.pivot(index="min users per track", columns="k", values="P@10").add_prefix("P@10, k=")
user_filter = vocabulary_info.set_index("min users per track")[
    ["tracks in vocabulary", "training rows dropped", "training rows dropped %", "hidden recommendable %"]]
user_filter.join(p10).round(4)

,tracks in vocabulary,training rows dropped,training rows dropped %,hidden recommendable %,"P@10, k=10","P@10, k=20","P@10, k=40","P@10, k=80"
min users per track,,,,,,,,
20,120650,6004246,31.5170,67.719,0.1257,0.1228,0.1202,0.1153
10,221287,4649529,24.4059,74.404,0.1286,0.1254,0.1213,0.1148
5,409289,3433359,18.0221,80.197,0.1292,0.1259,0.1241,0.1166
